In [ ]:
!pip install torch torchvision tqdm

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Tạo thư mục chứa dữ liệu sau khi giải nén
!mkdir /content/casia_data

# Giải nén file .rar
# -x: giải nén giữ nguyên cấu trúc thư mục
# -idq: chế độ "quiet" (không in tên 500.000 file ra màn hình để tránh treo trình duyệt)
!unrar x "/content/drive/MyDrive/database/casia.rar" /content/casia_data/ -idq

In [ ]:
import os

# Kiểm tra thư mục bên trong casia_data
base_path = '/content/casia_data/casia'
entities = os.listdir(base_path)
print(f"Số lượng thư mục người dùng (ID): {len(entities)}")

# Thử in ra 5 thư mục đầu tiên
print("Ví dụ 5 ID đầu tiên:", entities[:5])

Số lượng thư mục người dùng (ID): 10572
Ví dụ 5 ID đầu tiên: ['4619', '1135', '1545', '4125', '8001']


In [ ]:
# !pip uninstall torch -y
# !pip install torch torchvision

In [ ]:
import torch
print(torch.__version__)
print(torch.cuda.is_available())

2.10.0+cu128
True


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn import Parameter
import math
import os
import pandas as pd

class ConvBlock(nn.Module):
    def __init__(self, inp, oup, k, s, p, dw=False, linear=False):
        super().__init__()
        self.linear = linear
        if dw:
            self.conv = nn.Conv2d(inp, oup, k, s, p, groups=inp, bias=False)
        else:
            self.conv = nn.Conv2d(inp, oup, k, s, p, bias=False)
        self.bn = nn.BatchNorm2d(oup)
        if not linear:
            self.prelu = nn.PReLU(oup)

    def forward(self, x):
        x = self.bn(self.conv(x))
        return x if self.linear else self.prelu(x)

class Bottleneck(nn.Module):
    def __init__(self, inp, oup, stride, expansion):
        super().__init__()
        self.connect = stride == 1 and inp == oup
        self.conv = nn.Sequential(
            ConvBlock(inp, inp * expansion, 1, 1, 0),
            ConvBlock(inp * expansion, inp * expansion, 3, stride, 1, dw=True),
            ConvBlock(inp * expansion, oup, 1, 1, 0, linear=True),
        )

    def forward(self, x):
        return x + self.conv(x) if self.connect else self.conv(x)

class MobileFaceNet(nn.Module):
    def __init__(self, embedding_size=128):
        super().__init__()
        self.conv1 = ConvBlock(3, 64, 3, 2, 1)
        self.dw_conv1 = ConvBlock(64, 64, 3, 1, 1, dw=True)

        self.inplanes = 64
        setting = [
            [2, 64, 5, 2],
            [4, 128, 1, 2],
            [2, 128, 6, 1],
            [4, 128, 1, 2],
            [2, 128, 2, 1]
        ]

        layers = []
        for t, c, n, s in setting:
            for i in range(n):
                stride = s if i == 0 else 1
                layers.append(Bottleneck(self.inplanes, c, stride, t))
                self.inplanes = c
        self.blocks = nn.Sequential(*layers)

        self.conv2 = ConvBlock(128, 512, 1, 1, 0)
        self.linear7 = ConvBlock(512, 512, (7, 6), 1, 0, dw=True, linear=True)
        self.linear1 = ConvBlock(512, embedding_size, 1, 1, 0, linear=True)

        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm2d):
                m.weight.data.fill_(1)
                m.bias.data.zero_()

    def forward(self, x):
        x = self.conv1(x)
        x = self.dw_conv1(x)
        x = self.blocks(x)
        x = self.conv2(x)
        x = self.linear7(x)
        x = self.linear1(x)
        x = x.view(x.size(0), -1)
        return F.normalize(x)

class ArcMarginProduct(nn.Module):
    def __init__(self, in_features, out_features, s=32.0, m=0.50):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.s = s
        self.m = m
        self.weight = Parameter(torch.FloatTensor(out_features, in_features))
        nn.init.xavier_uniform_(self.weight)

        self.cos_m = math.cos(m)
        self.sin_m = math.sin(m)
        self.th = math.cos(math.pi - m)
        self.mm = math.sin(math.pi - m) * m

    def forward(self, input, label):
        cosine = F.linear(F.normalize(input), F.normalize(self.weight))
        sine = torch.sqrt(1.0 - torch.pow(cosine, 2))
        phi = cosine * self.cos_m - sine * self.sin_m
        phi = torch.where(cosine > self.th, phi, cosine - self.mm)

        one_hot = torch.zeros(cosine.size(), device=input.device)
        one_hot.scatter_(1, label.view(-1, 1).long(), 1)

        output = (one_hot * phi) + ((1.0 - one_hot) * cosine)
        output *= self.s
        return output

In [ ]:
import cv2
import numpy as np
from torch.utils.data import Dataset, DataLoader

class FaceDataset(Dataset):
    def __init__(self, root):
        self.image_paths = []
        self.labels = []
        self.class_to_idx = {}

        classes = sorted([d for d in os.listdir(root) if os.path.isdir(os.path.join(root, d))])
        for idx, person in enumerate(classes):
            self.class_to_idx[person] = idx
            person_path = os.path.join(root, person)
            for img_name in os.listdir(person_path):
                self.image_paths.append(os.path.join(person_path, img_name))
                self.labels.append(idx)
        self.num_classes = len(classes)

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        try:
            img = cv2.imread(self.image_paths[idx])
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            img = cv2.resize(img, (96, 112))
            img = (img - 127.5) / 128.0
            img = np.transpose(img, (2, 0, 1))
            return torch.tensor(img, dtype=torch.float32), torch.tensor(self.labels[idx], dtype=torch.long)
        except Exception as e:
            return self.__getitem__(0)

In [ ]:
def save_checkpoint(state, save_path, is_best=False):
    torch.save(state, save_path)
    if is_best:
        torch.save(state, save_path.replace('.pth', '_best.pth'))

def load_checkpoint(model, head, optimizer, checkpoint_path):
    if os.path.isfile(checkpoint_path):
        print(f"==> Loading checkpoint '{checkpoint_path}'")
        checkpoint = torch.load(checkpoint_path)
        start_epoch = checkpoint['epoch']
        model.load_state_dict(checkpoint['model_state_dict'])
        head.load_state_dict(checkpoint['head_state_dict'])
        # optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        history = checkpoint.get('history', [])
        print(f"==> Loaded checkpoint at epoch {start_epoch}")
        return start_epoch, history
    else:
        print(f"==> No checkpoint found at '{checkpoint_path}'")
        return 0, []

DATA_ROOT = "/content/casia_data/casia"
CHECKPOINT_DIR = "/content/drive/MyDrive/face_model_checkpoints"
LOG_FILE = os.path.join(CHECKPOINT_DIR, "train_log.csv")
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dataset = FaceDataset(DATA_ROOT)
loader = DataLoader(dataset, batch_size=128, shuffle=True, num_workers=2, pin_memory=True)

backbone = MobileFaceNet(embedding_size=128).to(device)
head = ArcMarginProduct(128, dataset.num_classes).to(device)

optimizer = torch.optim.SGD([
    {'params': backbone.parameters()},
    {'params': head.parameters()}
], lr=0.1, momentum=0.9, weight_decay=5e-4)

criterion = nn.CrossEntropyLoss()

checkpoint_path = os.path.join(CHECKPOINT_DIR, "last_checkpoint.pth")
start_epoch, history = load_checkpoint(backbone, head, optimizer, checkpoint_path)

for g in optimizer.param_groups:
    g['lr'] = 0.0003

scheduler = torch.optim.lr_scheduler.MultiStepLR(
    optimizer,
    milestones=[58, 62],
    gamma=0.1,
    # last_epoch=start_epoch - 1
)
from tqdm import tqdm

TOTAL_EPOCHS = 65

for epoch in range(start_epoch, TOTAL_EPOCHS):
    backbone.train()
    head.train()
    total_loss = 0

    pbar = tqdm(loader, desc=f"Epoch {epoch+1}/{TOTAL_EPOCHS}")
    for imgs, labels in pbar:
        imgs, labels = imgs.to(device), labels.to(device)

        features = backbone(imgs)
        logits = head(features, labels)
        loss = criterion(logits, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        pbar.set_postfix(loss=loss.item())

    avg_loss = total_loss / len(loader)
    print(f"Epoch {epoch+1} Average Loss: {avg_loss:.4f}")

    history.append({'epoch': epoch + 1, 'loss': avg_loss})
    pd.DataFrame(history).to_csv(LOG_FILE, index=False)

    checkpoint_state = {
        'epoch': epoch + 1,
        'model_state_dict': backbone.state_dict(),
        'head_state_dict': head.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'history': history
    }
    save_checkpoint(checkpoint_state, checkpoint_path)
    scheduler.step()

    # if (epoch + 1) in [20, 35, 45]:
    #     for g in optimizer.param_groups:FF
    #         g['lr'] *= 0.1
    print(f"Epoch {epoch+1} - LR: {optimizer.param_groups[0]['lr']}")

print("Training hoàn tất!")

==> Loading checkpoint '/content/drive/MyDrive/face_model_checkpoints/last_checkpoint.pth'
==> Loaded checkpoint at epoch 58


Epoch 59/65: 100%|██████████| 3833/3833 [13:05<00:00,  4.88it/s, loss=5.1]


Epoch 59 Average Loss: 5.6991
Epoch 59 - LR: 0.0003


Epoch 60/65: 100%|██████████| 3833/3833 [13:05<00:00,  4.88it/s, loss=5.86]


Epoch 60 Average Loss: 5.6664
Epoch 60 - LR: 0.0003


Epoch 61/65:  29%|██▉       | 1104/3833 [03:46<09:15,  4.91it/s, loss=5.37]